# Inverse optimization at one held-out operating point

Run from `notebooks/` after notebooks 01 and 02. The frozen surrogate optimizes five controls with fixed test disturbances. A deterministic random draw selects a test observation whose current controls and disturbances are within a training-derived support envelope. The actuator constraints in `src/constraints.py` use dimensionless commands: map the training 1st–99th percentile range of each physical control to commands 0–1, then enforce actuator bounds and a maximum command change of 0.05. The bounds do **not** directly apply to the physical CSV columns.

The output limits below are an illustrative quality floor and energy ceiling relative to the starting model prediction. Penalties discourage output violations but do not guarantee output feasibility. The plant comparison uses the *same fixed disturbances* and the original noise-free **steady-state** generator equation. It is not an observed or instantaneous improvement: the data include lag and measurement noise, and the learned surrogate is static.

In [ ]:
from pathlib import Path
import ast
import pickle
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import torch
from scipy.special import expit
from torch import nn

sys.path.insert(0, str(Path('../src').resolve()))
from constraints import CONTROL_NAMES, SigmoidControlParameterization
from objective_functions import InverseProcessObjective

SEED = 42
STEPS = 250
LEARNING_RATE = 0.06
DATA_PATH = Path('../data/synthetic_process_data.csv')
MODEL_DIR = Path('../models')
RESULTS_DIR = Path('../results')
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
checkpoint = torch.load(MODEL_DIR / 'process_model.pt', weights_only=True, map_location='cpu')
# Pickle can execute code: load only scalers created locally by notebook 02.
with (MODEL_DIR / 'input_scaler.pkl').open('rb') as file:
    input_scaler = pickle.load(file)
with (MODEL_DIR / 'output_scaler.pkl').open('rb') as file:
    output_scaler = pickle.load(file)
input_columns = checkpoint['input_columns']
output_columns = checkpoint['output_columns']
if (input_columns[5:] != list(CONTROL_NAMES) or len(input_columns) != 10
        or output_columns != ['y1_quality', 'y2_energy_load']
        or tuple(checkpoint['hidden_sizes']) != (32, 32, 16)):
    raise ValueError('Unexpected checkpoint architecture or column order')
if (list(input_scaler.feature_names_in_) != input_columns
        or list(output_scaler.feature_names_in_) != output_columns):
    raise ValueError('Scaler column order differs from the checkpoint')

class ProcessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32), nn.SiLU(), nn.Linear(32, 32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU(), nn.Linear(16, 2),
        )

    def forward(self, inputs):
        return self.network(inputs)

model = ProcessModel()
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval().requires_grad_(False)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Run notebook 01 first: {DATA_PATH.resolve()}')
data = pd.read_csv(DATA_PATH)
if len(data) != 8000 or not np.isfinite(data[input_columns + output_columns].to_numpy(dtype=float)).all():
    raise ValueError('Expected 8,000 finite observations with the checkpoint columns')
train = data.iloc[:int(0.70 * len(data))]
test = data.iloc[int(0.85 * len(data)):]
if (not np.allclose(input_scaler.mean_, train[input_columns].mean().to_numpy(), rtol=1e-5)
        or not np.allclose(output_scaler.mean_, train[output_columns].mean().to_numpy(), rtol=1e-5)):
    raise ValueError('Scalers do not match the chronological training data')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
support_lower = train[input_columns].quantile(0.01).to_numpy(dtype=np.float32)
support_upper = train[input_columns].quantile(0.99).to_numpy(dtype=np.float32)
control_origin = support_lower[5:]
control_span = support_upper[5:] - control_origin
parameterization = SigmoidControlParameterization()
test_inputs = test[input_columns].to_numpy(dtype=np.float32)
test_commands = (test_inputs[:, 5:] - control_origin) / control_span
eligible = np.flatnonzero(
    np.all((test_inputs[:, :5] >= support_lower[:5]) & (test_inputs[:, :5] <= support_upper[:5]), axis=1)
    & np.all((test_commands >= parameterization.lower.numpy())
             & (test_commands <= parameterization.upper.numpy()), axis=1)
)
if not len(eligible):
    raise ValueError('No held-out operating point satisfies initial actuator and support limits')
selected = int(rng.choice(eligible))
sample_index = int(test.index[selected])
disturbances = torch.from_numpy(test_inputs[selected, :5].copy())
initial_controls = torch.from_numpy(test_inputs[selected, 5:].copy())
previous_commands = torch.from_numpy(test_commands[selected].copy())
origin = torch.from_numpy(control_origin.copy())
span = torch.from_numpy(control_span.copy())
control_lower = origin + span * parameterization.lower
control_upper = origin + span * parameterization.upper
print(f'Selected test sample {sample_index} from {len(eligible)} feasible candidates')
display(pd.DataFrame({'disturbance': input_columns[:5], 'fixed_value': disturbances.numpy()}))


In [ ]:
def predict_outputs(controls):
    full = torch.cat((disturbances, controls))
    scaled = (full - torch.as_tensor(input_scaler.mean_, dtype=torch.float32)) / torch.as_tensor(input_scaler.scale_, dtype=torch.float32)
    return (model(scaled.unsqueeze(0)).squeeze(0)
            * torch.as_tensor(output_scaler.scale_, dtype=torch.float32)
            + torch.as_tensor(output_scaler.mean_, dtype=torch.float32))

with torch.no_grad():
    initial_prediction = predict_outputs(initial_controls)
# Explicit process targets, defined relative to the starting prediction.
QUALITY_MIN = float(initial_prediction[0] - 0.25)
ENERGY_MAX = float(initial_prediction[1] + 0.25)
objective = InverseProcessObjective(
    model,
    torch.as_tensor(input_scaler.mean_, dtype=torch.float32),
    torch.as_tensor(input_scaler.scale_, dtype=torch.float32),
    torch.as_tensor(output_scaler.mean_, dtype=torch.float32),
    torch.as_tensor(output_scaler.scale_, dtype=torch.float32),
    control_lower, control_upper,
    torch.from_numpy(support_lower), torch.from_numpy(support_upper),
    quality_min=QUALITY_MIN, energy_max=ENERGY_MAX,
    quality_weight=1.0, energy_weight=1.0,
    movement_weight=0.1, constraint_weight=20.0, support_weight=10.0,
)
print(f'Output limits: quality >= {QUALITY_MIN:.3f}; energy <= {ENERGY_MAX:.3f}')

def command_to_physical(commands):
    return origin + span * commands

def diagnostics(commands, controls, outputs, loss):
    moves = commands - previous_commands
    return {
        'objective': float(loss.detach()),
        'predicted_quality': float(outputs[0].detach()),
        'predicted_energy': float(outputs[1].detach()),
        'quality_shortfall': max(QUALITY_MIN - float(outputs[0].detach()), 0.0),
        'energy_excess': max(float(outputs[1].detach()) - ENERGY_MAX, 0.0),
        'max_rate_violation': float(torch.relu(moves.abs() - parameterization.max_move).max().detach()),
        'max_bound_violation': float(torch.maximum(torch.relu(parameterization.lower - commands),
                                                    torch.relu(commands - parameterization.upper)).max().detach()),
        'max_support_violation': float(torch.maximum(
            torch.relu((objective.support_lower - torch.cat((disturbances, controls))) / objective.input_scale),
            torch.relu((torch.cat((disturbances, controls)) - objective.support_upper) / objective.input_scale)
        ).max().detach()),
        'max_command_move': float(moves.abs().max().detach()),
        **{f'move_{name}': float(moves[i].detach()) for i, name in enumerate(CONTROL_NAMES)},
    }

# A zero logit is the middle of each feasible rate interval: at the measured starting point.
logits = nn.Parameter(torch.zeros(5))
optimizer = torch.optim.Adam([logits], lr=LEARNING_RATE)
history = []
for step in range(STEPS + 1):
    optimizer.zero_grad(set_to_none=True)
    commands = parameterization(logits, previous_commands)
    controls = command_to_physical(commands)
    loss = objective(disturbances, controls, initial_controls)
    outputs = predict_outputs(controls)
    row = diagnostics(commands, controls, outputs, loss)
    history.append({'step': step, **row})
    if step < STEPS:
        loss.backward()
        if not torch.isfinite(logits.grad).all():
            raise ValueError(f'Non-finite optimizer gradient at step {step}')
        optimizer.step()
history = pd.DataFrame(history)
history.to_csv(RESULTS_DIR / 'inverse_optimization_history.csv', index=False)
with torch.no_grad():
    optimal_commands = parameterization(logits, previous_commands)
    optimal_controls = command_to_physical(optimal_commands)
    optimal_prediction = predict_outputs(optimal_controls)
assert torch.all(optimal_commands >= parameterization.lower) and torch.all(optimal_commands <= parameterization.upper)
assert torch.all((optimal_commands - previous_commands).abs() <= parameterization.max_move + 1e-6)
print('Last optimization step:')
display(history.tail(1).T)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
axes[0, 0].plot(history['step'], history['objective'])
axes[0, 0].set(xlabel='Adam step', ylabel='Scalar loss', title='Objective')
axes[0, 1].plot(history['step'], history['predicted_quality'], label='Quality')
axes[0, 1].plot(history['step'], history['predicted_energy'], label='Energy')
axes[0, 1].axhline(QUALITY_MIN, color='C0', linestyle='--', label='Quality minimum')
axes[0, 1].axhline(ENERGY_MAX, color='C1', linestyle='--', label='Energy maximum')
axes[0, 1].set(xlabel='Adam step', ylabel='Predicted output', title='Outputs and limits')
axes[0, 1].legend(fontsize=8)
for name in CONTROL_NAMES:
    axes[1, 0].plot(history['step'], history[f'move_{name}'], label=name)
axes[1, 0].axhline(0.05, color='k', linestyle='--', linewidth=1)
axes[1, 0].axhline(-0.05, color='k', linestyle='--', linewidth=1)
axes[1, 0].set(xlabel='Adam step', ylabel='Change in normalized command', title='Control movements')
axes[1, 0].legend(fontsize=7)
for name in ['quality_shortfall', 'energy_excess', 'max_rate_violation',
             'max_bound_violation', 'max_support_violation']:
    axes[1, 1].plot(history['step'], history[name], label=name)
axes[1, 1].set(xlabel='Adam step', ylabel='Violation (own scaled units)', title='Constraint and support diagnostics')
axes[1, 1].legend(fontsize=7)
fig.savefig(RESULTS_DIR / 'inverse_optimization_convergence.png', dpi=150)
plt.show()


## Predicted versus original process response

Load only the `steady_state_response` function definition from the **trusted local** data-generation notebook, so the plant equation has one source of truth and the data-generation and plotting cells are not rerun. Its result is a noise-free steady-state counterfactual at identical disturbances. The observed CSV output at the sampled time is shown separately because it reflects prior controls, lag and noise.

In [ ]:
generator_path = Path('01_generate_data.ipynb')
generator = nbformat.read(generator_path, as_version=4)
definitions = [ast.parse(''.join(cell.source)) for cell in generator.cells
               if cell.cell_type == 'code' and 'def steady_state_response(' in ''.join(cell.source)]
if len(definitions) != 1:
    raise ValueError('Could not uniquely locate the original steady-state plant function')
functions = [node for node in definitions[0].body
             if isinstance(node, ast.FunctionDef) and node.name == 'steady_state_response']
if len(functions) != 1:
    raise ValueError('Original plant function was not found')
plant_scope = {'np': np, 'expit': expit}
exec(compile(ast.Module(body=functions, type_ignores=[]), str(generator_path), 'exec'), plant_scope)
plant_response = plant_scope['steady_state_response']
fixed_disturbances = disturbances.numpy()[None, :]
true_start = plant_response(fixed_disturbances, initial_controls.numpy()[None, :])[0]
true_optimum = plant_response(fixed_disturbances, optimal_controls.numpy()[None, :])[0]
pred_start = initial_prediction.numpy()
pred_optimum = optimal_prediction.numpy()
comparison = pd.DataFrame({
    'output': output_columns,
    'observed_start_lagged_noisy': test.iloc[selected][output_columns].to_numpy(dtype=float),
    'predicted_start': pred_start,
    'predicted_optimum': pred_optimum,
    'predicted_change': pred_optimum - pred_start,
    'true_steady_state_start': true_start,
    'true_steady_state_optimum': true_optimum,
    'true_steady_state_change': true_optimum - true_start,
})
controls_table = pd.DataFrame({
    'control': CONTROL_NAMES,
    'starting_physical': initial_controls.numpy(),
    'optimal_physical': optimal_controls.numpy(),
    'physical_move': (optimal_controls - initial_controls).numpy(),
    'starting_command': previous_commands.numpy(),
    'optimal_command': optimal_commands.numpy(),
    'command_move': (optimal_commands - previous_commands).numpy(),
})
comparison.to_csv(RESULTS_DIR / 'inverse_optimization_comparison.csv', index=False)
controls_table.to_csv(RESULTS_DIR / 'inverse_optimization_controls.csv', index=False)
print(f'Fixed test sample: {sample_index}; objective: {history.objective.iloc[0]:.4f} -> {history.objective.iloc[-1]:.4f}')
display(controls_table)
display(comparison)
print('Final violations (positive means unmet constraint):')
display(history[['quality_shortfall', 'energy_excess', 'max_rate_violation',
                 'max_bound_violation', 'max_support_violation']].tail(1))
